Silver layer para a base de motivos.

padronizando strings categoricas usando `lower()`e `F.trim()` .

Além de mapeando categorias nulas.
#TODO: type safety e write.


In [0]:
from config import BRONZE_SCHEMA,SILVER_SCHEMA, CATALOG
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
motivos_bronze = spark.table(f"{BRONZE_SCHEMA}.base_motivos")
motivos_bronze.printSchema()
display(motivos_bronze)

In [0]:
categoria_por_id = {
    14:'Cadastral',	#Transferência de agência
    16: 'Cadastral', #Cancelamento de conta
    12: 'Cartao' #Contratacao de cartão adicional 
}
categoria_corrigida = F.col('categoria')
for id, categoria in categoria_por_id.items():
    categoria_corrigida= F.when(
        F.col('id_motivo') == id ,
        F.lit(categoria),
    ).otherwise(categoria_corrigida)
motivos_silver =motivos_bronze.withColumn('categoria', categoria_corrigida)
#removendo espaços em branco e padronizando as strings categoricas em lower(isso não inclue o nome_motivo)
motivos_silver = motivos_silver.withColumn('criticidade',F.lower(F.trim(F.col('criticidade'))))
motivos_silver = motivos_silver.withColumn('categoria',F.lower(F.trim(F.col('categoria'))))
motivos_silver = motivos_silver.withColumn('nome_motivo',F.trim(F.col('nome_motivo')))

In [0]:
#TODO: fazer isso gerar um erro em caso de fail.
_,passou = dq_check('motivos_bronze','id nulos',motivos_silver,F.col('id_motivo').isNotNull())
if not passou :
    raise ValueError('valores de id_motivo nulos')
_,passou =dq_check_unique('motivos_bronze','id unicos',motivos_silver,['id_motivo'])
if not passou :
    raise ValueError('valores de id_motivo nulos')

In [0]:
_,passou = dq_check('motivos_silver','nome_motivo nulo',motivos_silver,F.col('nome_motivo').isNotNull())
if not passou :
    raise ValueError('valores de id_motivo nulos')
dq_check('motivos_silver','categoria nulas',motivos_silver,F.col('categoria').isNotNull())
if not passou :
    raise ValueError('valores de id_motivo nulos')
dq_check_unique('motivos_silver','id nulos',motivos_silver,['id_motivo'])
if not passou :
    raise ValueError('valores de id_motivo nulos')

criticidade_validas = ['alta','media','baixa']
categorias_validas = ['cartao','financeiro','beneficios','atendimento','cadastral']


df,passou=dq_check('motivos_silver','categorias validas',motivos_silver,F.col('categoria').isNull()
         | F.col('categoria').isin(categorias_validas))
if not passou :
    raise ValueError('categorias  invalidas')

df,passou=dq_check('motivos_silver','criticidade validas',motivos_silver,F.col('criticidade').isNull()
         | F.col('criticidade').isin(criticidade_validas))
if not passou :
    raise ValueError('criticidade invalidas')


In [0]:
motivos_silver.write.format('delta').mode('overwrite').saveAsTable(f"{SILVER_SCHEMA}.base_motivos")